# Gold Layer — Yelp Businesses (Nested Flatten)

## Transformations
| Source field | Type | Gold output |
|---|---|---|
| `attributes` | `dict` (35 keys, một số là string-encoded Python dict) | 35+ flat columns |
| `hours` | `dict` (7 keys) | `hours_monday` … `hours_sunday` |
| `categories` | CSV string | `array<string>` |
| Còn lại | scalar | giữ nguyên |

Target table: `nessie.gold.businesses_flat`

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Yelp_Gold_Businesses") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.io.ResolvingFileIO") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.defaultCatalog", "nessie") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

hc = spark.sparkContext._jsc.hadoopConfiguration()
hc.set("fs.s3a.endpoint",               "http://minio:9000")
hc.set("fs.s3a.access.key",             "admin")
hc.set("fs.s3a.secret.key",             "password")
hc.set("fs.s3a.path.style.access",      "true")
hc.set("fs.s3a.connection.ssl.enabled", "false")
hc.set("fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession ready")

✅ SparkSession ready


26/06/16 02:14:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Step 1 — Đọc raw business.json vào Spark

In [2]:
BUSINESS_PATH = "/home/jovyan/data/yelp/yelp_academic_dataset_business.json"

# Spark đọc JSONL (mỗi dòng là 1 JSON object)
raw_df = spark.read.json(BUSINESS_PATH)

print(f"Total businesses : {raw_df.count():,}")
print(f"Total columns    : {len(raw_df.columns)}")
print()
raw_df.printSchema()

Total businesses : 150,346
Total columns    : 14

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: stri

In [3]:
# Xem 1 sample record để confirm schema
import json
sample = raw_df.limit(1).toPandas().to_dict(orient="records")[0]
print(json.dumps(
    {k: str(v)[:120] for k, v in sample.items()},
    indent=2, ensure_ascii=False
))

{
  "address": "1616 Chapala St, Ste 2",
  "attributes": "Row(AcceptsInsurance=None, AgesAllowed=None, Alcohol=None, Ambience=None, BYOB=None, BYOBCorkage=None, BestNights=None, ",
  "business_id": "Pns2l4eNsfO8kk83dixA6A",
  "categories": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists",
  "city": "Santa Barbara",
  "hours": "None",
  "is_open": "0",
  "latitude": "34.4266787",
  "longitude": "-119.7111968",
  "name": "Abby Rappoport, LAC, CMQ",
  "postal_code": "93101",
  "review_count": "7",
  "stars": "5.0",
  "state": "CA"
}


## Step 2 — Định nghĩa UDFs để flatten nested fields

In [4]:
import ast
from pyspark.sql.functions import udf, col, split, when, lit
from pyspark.sql.types import (
    StringType, BooleanType, ArrayType, MapType, FloatType
)

# ── UDF 1: Parse string-encoded Python dict ──────────────────────
# Ví dụ: "{'garage': False, 'street': True}" → {'garage': False, 'street': True}
# KHÔNG dùng json.loads vì Python dùng True/False/None thay vì true/false/null
def safe_parse_python_dict(s):
    if s is None or s == 'None':
        return None
    try:
        return {str(k): str(v) for k, v in ast.literal_eval(s).items()}
    except:
        return None

parse_dict_udf = udf(safe_parse_python_dict, MapType(StringType(), StringType()))

# ── UDF 2: Extract 1 key từ string-encoded Python dict ───────────
def extract_from_python_dict(s, key):
    if s is None or s == 'None':
        return None
    try:
        d = ast.literal_eval(s)
        v = d.get(key)
        return str(v) if v is not None else None
    except:
        return None

# ── UDF 3: Parse categories CSV string → array ───────────────────
def parse_categories(s):
    if s is None:
        return []
    return [c.strip() for c in s.split(",") if c.strip()]

parse_categories_udf = udf(parse_categories, ArrayType(StringType()))

# ── UDF 4: Normalize boolean string ──────────────────────────────
# Yelp dùng 'True'/'False'/'None' (Python strings)
def to_bool(s):
    if s is None or s == 'None':
        return None
    return s.strip().lower() == 'true'

to_bool_udf = udf(to_bool, BooleanType())

print("✅ UDFs registered")

✅ UDFs registered


## Step 3 — Flatten tất cả nested fields

In [5]:
from pyspark.sql.functions import col

# ── Scalar fields (giữ nguyên) ────────────────────────────────────
flat_df = raw_df.select(
    # Identity
    col("business_id"),
    col("name"),
    col("address"),
    col("city"),
    col("state"),
    col("postal_code"),
    col("latitude"),
    col("longitude"),
    col("stars"),
    col("review_count"),
    col("is_open"),

    # Categories: CSV → array<string>
    parse_categories_udf(col("categories")).alias("categories"),

    # ── attributes: scalar booleans/strings ──────────────────────
    to_bool_udf(col("attributes.RestaurantsTakeOut")).alias("attr_restaurants_takeout"),
    to_bool_udf(col("attributes.RestaurantsDelivery")).alias("attr_restaurants_delivery"),
    to_bool_udf(col("attributes.RestaurantsReservations")).alias("attr_restaurants_reservations"),
    to_bool_udf(col("attributes.OutdoorSeating")).alias("attr_outdoor_seating"),
    to_bool_udf(col("attributes.WiFi")).alias("attr_wifi_raw"),  # 'free'/'paid'/'no'
    to_bool_udf(col("attributes.BikeParking")).alias("attr_bike_parking"),
    to_bool_udf(col("attributes.WheelchairAccessible")).alias("attr_wheelchair_accessible"),
    to_bool_udf(col("attributes.HappyHour")).alias("attr_happy_hour"),
    to_bool_udf(col("attributes.GoodForKids")).alias("attr_good_for_kids"),
    to_bool_udf(col("attributes.DogsAllowed")).alias("attr_dogs_allowed"),
    to_bool_udf(col("attributes.HasTV")).alias("attr_has_tv"),
    to_bool_udf(col("attributes.RestaurantsGoodForGroups")).alias("attr_good_for_groups"),
    to_bool_udf(col("attributes.Caters")).alias("attr_caters"),
    col("attributes.RestaurantsPriceRange2").alias("attr_price_range"),
    col("attributes.NoiseLevel").alias("attr_noise_level"),
    col("attributes.Alcohol").alias("attr_alcohol"),
    col("attributes.RestaurantsAttire").alias("attr_attire"),

    # ── attributes: string-encoded Python dicts (2nd-level parse) ─
    # BusinessParking
    parse_dict_udf(col("attributes.BusinessParking")).alias("attr_parking_map"),
    # Ambience
    parse_dict_udf(col("attributes.Ambience")).alias("attr_ambience_map"),
    # GoodForMeal
    parse_dict_udf(col("attributes.GoodForMeal")).alias("attr_good_for_meal_map"),
    # Music
    parse_dict_udf(col("attributes.Music")).alias("attr_music_map"),
    # BestNights
    parse_dict_udf(col("attributes.BestNights")).alias("attr_best_nights_map"),

    # ── hours: dict → 7 columns ───────────────────────────────────
    col("hours.Monday").alias("hours_monday"),
    col("hours.Tuesday").alias("hours_tuesday"),
    col("hours.Wednesday").alias("hours_wednesday"),
    col("hours.Thursday").alias("hours_thursday"),
    col("hours.Friday").alias("hours_friday"),
    col("hours.Saturday").alias("hours_saturday"),
    col("hours.Sunday").alias("hours_sunday"),
)

print(f"Columns sau flatten: {len(flat_df.columns)}")
print(flat_df.columns)

Columns sau flatten: 41
['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'categories', 'attr_restaurants_takeout', 'attr_restaurants_delivery', 'attr_restaurants_reservations', 'attr_outdoor_seating', 'attr_wifi_raw', 'attr_bike_parking', 'attr_wheelchair_accessible', 'attr_happy_hour', 'attr_good_for_kids', 'attr_dogs_allowed', 'attr_has_tv', 'attr_good_for_groups', 'attr_caters', 'attr_price_range', 'attr_noise_level', 'attr_alcohol', 'attr_attire', 'attr_parking_map', 'attr_ambience_map', 'attr_good_for_meal_map', 'attr_music_map', 'attr_best_nights_map', 'hours_monday', 'hours_tuesday', 'hours_wednesday', 'hours_thursday', 'hours_friday', 'hours_saturday', 'hours_sunday']


In [6]:
# ── Step 3b: Expand string-encoded dict fields thành sub-columns ──
# Mỗi Map field ở trên → extract từng key thành column riêng

from pyspark.sql.functions import col

def expand_map_col(df, map_col, prefix, keys):
    """Extract từng key từ MapType column thành individual boolean columns."""
    for key in keys:
        df = df.withColumn(
            f"{prefix}_{key}",
            to_bool_udf(col(map_col).getItem(key))
        )
    return df

# BusinessParking keys
flat_df = expand_map_col(flat_df, "attr_parking_map", "parking",
    ["garage", "street", "validated", "lot", "valet"])

# Ambience keys
flat_df = expand_map_col(flat_df, "attr_ambience_map", "ambience",
    ["touristy", "hipster", "romantic", "divey", "intimate",
     "trendy", "upscale", "classy", "casual"])

# GoodForMeal keys
flat_df = expand_map_col(flat_df, "attr_good_for_meal_map", "meal",
    ["dessert", "latenight", "lunch", "dinner", "brunch", "breakfast"])

# Music keys
flat_df = expand_map_col(flat_df, "attr_music_map", "music",
    ["dj", "background_music", "no_music", "jukebox", "live", "video", "karaoke"])

# BestNights keys
flat_df = expand_map_col(flat_df, "attr_best_nights_map", "best_night",
    ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"])

# Drop map columns sau khi đã expand
flat_df = flat_df.drop(
    "attr_parking_map", "attr_ambience_map",
    "attr_good_for_meal_map", "attr_music_map", "attr_best_nights_map"
)

print(f"Columns sau expand: {len(flat_df.columns)}")
flat_df.printSchema()

Columns sau expand: 70
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- is_open: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- attr_restaurants_takeout: boolean (nullable = true)
 |-- attr_restaurants_delivery: boolean (nullable = true)
 |-- attr_restaurants_reservations: boolean (nullable = true)
 |-- attr_outdoor_seating: boolean (nullable = true)
 |-- attr_wifi_raw: boolean (nullable = true)
 |-- attr_bike_parking: boolean (nullable = true)
 |-- attr_wheelchair_accessible: boolean (nullable = true)
 |-- attr_happy_hour: boolean (nullable = true)
 |-- attr_good_for_

## Step 4 — Ghi vào Gold Iceberg table

In [7]:
# Tạo namespace gold
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

# Ghi batch vào Iceberg (Gold là batch, không streaming)
print("[*] Đang ghi vào nessie.gold.businesses_flat...")

flat_df.writeTo("nessie.gold.businesses_flat") \
    .using("iceberg") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()

count = spark.sql("SELECT COUNT(*) FROM nessie.gold.businesses_flat").collect()[0][0]
print(f"✅ Ghi xong! Total records: {count:,}")

[*] Đang ghi vào nessie.gold.businesses_flat...


✅ Ghi xong! Total records: 150,346


## Step 5 — Validation

In [8]:
# Validation 1: Schema tổng quan
print("=" * 55)
print("VALIDATION 1: TABLE OVERVIEW")
print("=" * 55)

gold = spark.table("nessie.gold.businesses_flat")
total = gold.count()
null_attrs = gold.filter(col("attr_restaurants_takeout").isNull()).count()
null_hours = gold.filter(col("hours_monday").isNull()).count()

print(f"  Total businesses       : {total:,}")
print(f"  Total columns          : {len(gold.columns)}")
print(f"  Null attr_takeout      : {null_attrs:,} ({null_attrs/total*100:.1f}%)")
print(f"  Null hours_monday      : {null_hours:,} ({null_hours/total*100:.1f}%)")

VALIDATION 1: TABLE OVERVIEW


  Total businesses       : 150,346
  Total columns          : 70
  Null attr_takeout      : 92,594 (61.6%)
  Null hours_monday      : 35,872 (23.9%)


In [9]:
# Validation 2: Categories được parse đúng
print("=" * 55)
print("VALIDATION 2: CATEGORIES PARSING")
print("=" * 55)

from pyspark.sql.functions import size, explode

spark.sql("""
    SELECT name, categories, size(categories) as num_cats
    FROM nessie.gold.businesses_flat
    WHERE size(categories) > 3
    ORDER BY num_cats DESC
    LIMIT 5
""").show(truncate=False)

# Top 10 categories phổ biến nhất
print("\nTop 10 categories:")
spark.sql("""
    SELECT cat, COUNT(*) as cnt
    FROM nessie.gold.businesses_flat
    LATERAL VIEW explode(categories) tmp AS cat
    GROUP BY cat
    ORDER BY cnt DESC
    LIMIT 10
""").show()

VALIDATION 2: CATEGORIES PARSING


+-----------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|name                                     |categories                                                                                                                                                                                                                                                                                                                                                                                                    

+----------------+-----+
|             cat|  cnt|
+----------------+-----+
|     Restaurants|52268|
|            Food|27781|
|        Shopping|24395|
|   Home Services|14356|
|   Beauty & Spas|14292|
|       Nightlife|12281|
|Health & Medical|11890|
|  Local Services|11198|
|            Bars|11065|
|      Automotive|10773|
+----------------+-----+



In [10]:
# Validation 3: Nested dict fields được flatten đúng
print("=" * 55)
print("VALIDATION 3: NESTED DICT FLATTEN")
print("=" * 55)

spark.sql("""
    SELECT
        name,
        attr_restaurants_takeout,
        attr_outdoor_seating,
        parking_garage,
        parking_street,
        ambience_casual,
        ambience_romantic,
        meal_lunch,
        meal_dinner,
        hours_monday,
        hours_friday
    FROM nessie.gold.businesses_flat
    WHERE parking_garage IS NOT NULL
      AND hours_monday IS NOT NULL
    LIMIT 5
""").show(truncate=False)

VALIDATION 3: NESTED DICT FLATTEN
+------------------+------------------------+--------------------+--------------+--------------+---------------+-----------------+----------+-----------+------------+------------+
|name              |attr_restaurants_takeout|attr_outdoor_seating|parking_garage|parking_street|ambience_casual|ambience_romantic|meal_lunch|meal_dinner|hours_monday|hours_friday|
+------------------+------------------------+--------------------+--------------+--------------+---------------+-----------------+----------+-----------+------------+------------+
|Target            |false                   |false               |false         |false         |NULL           |NULL             |NULL      |NULL       |8:0-22:0    |8:0-23:0    |
|St Honore Pastries|true                    |false               |false         |true          |NULL           |NULL             |NULL      |NULL       |7:0-20:0    |7:0-21:0    |
|Famous Footwear   |NULL                    |NULL                |

In [11]:
# Validation 4: Analytics query — chứng minh Gold layer dùng được
print("=" * 55)
print("VALIDATION 4: ANALYTICS QUERY")
print("=" * 55)

# Top 10 states theo số lượng restaurants mở cửa, có outdoor seating
spark.sql("""
    SELECT
        state,
        COUNT(*) AS total_businesses,
        ROUND(AVG(stars), 2) AS avg_stars,
        SUM(CAST(attr_outdoor_seating AS INT)) AS with_outdoor_seating,
        SUM(CAST(attr_restaurants_takeout AS INT)) AS with_takeout
    FROM nessie.gold.businesses_flat
    WHERE is_open = 1
    GROUP BY state
    ORDER BY total_businesses DESC
    LIMIT 10
""").show()

# Business có đủ thứ nhất: parking, outdoor, wifi, delivery
print("\nBusiness 'all-inclusive' (có tất cả tiện ích):")
spark.sql("""
    SELECT name, city, state, stars, review_count
    FROM nessie.gold.businesses_flat
    WHERE attr_outdoor_seating = true
      AND attr_restaurants_delivery = true
      AND parking_lot = true
      AND is_open = 1
    ORDER BY stars DESC, review_count DESC
    LIMIT 10
""").show(truncate=False)

VALIDATION 4: ANALYTICS QUERY
+-----+----------------+---------+--------------------+------------+
|state|total_businesses|avg_stars|with_outdoor_seating|with_takeout|
+-----+----------------+---------+--------------------+------------+
|   PA|           26289|      3.6|                2814|        8619|
|   FL|           21540|     3.63|                2864|        6231|
|   TN|            9600|     3.58|                1323|        3083|
|   IN|            8946|      3.6|                1105|        3071|
|   MO|            8363|     3.57|                1219|        2832|
|   AZ|            8108|     3.62|                 930|        1869|
|   LA|            7676|      3.7|                1134|        2441|
|   NJ|            7031|     3.47|                 678|        2652|
|   NV|            6277|     3.77|                 492|        1175|
|   AB|            4346|     3.46|                 492|        1737|
+-----+----------------+---------+--------------------+------------+


Bu

In [12]:
# Iceberg snapshots — xem lịch sử ghi
print("=" * 55)
print("ICEBERG SNAPSHOTS — Gold Layer")
print("=" * 55)

spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM nessie.gold.businesses_flat.snapshots
    ORDER BY committed_at
""").show(truncate=False)

ICEBERG SNAPSHOTS — Gold Layer
+-------------------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                     